# 11.5 - Comprehensions => Performance, Readability & Best Practices

## Why Comprehensions Are Usually Faster

- A comprehension appends with a dedicated bytecode instruction instead of looking up and calling `.append()` on every iteration.
- Since Python 3.12, list, dict and set comprehensions inside functions are inlined (PEP 709), which removes the cost of a hidden function call.

Speed is a bonus, not the main reason to use them. **Readability comes first.**

## Always Measure

Use `timeit` to compare versions on your own data:

```python
import timeit
timeit.timeit("[n * n for n in range(1000)]", number=1000)
```

Results depend on the machine, the Python version and the data.

## Memory

- A **list comprehension** stores every item.
- A **generator expression** stores only its current state.

For a single pass over a large sequence, prefer the generator:

```python
sum(n * n for n in range(10_000_000))    # no big list in memory
```

## Choosing the Right Tool

| Situation | Use |
|---|---|
| Build a new list, set or dict from an iterable | Comprehension |
| Consume values once (`sum`, `any`, `max`, `join`) | Generator expression |
| Values are needed more than once, or need `len()` / indexing | List comprehension |
| Side effects (printing, writing, updating) | Regular `for` loop |
| More than two `for` clauses or complex conditions | Loop or helper function |
| Chain, group, batch or combine iterators | `itertools` (Part 12) |

## Readability Rules

1. One idea per comprehension.
2. At most two `for` clauses.
3. Keep the condition short. Move complex tests into a named function.
4. Use clear names, not just `x`, when the meaning matters.
5. Split long comprehensions over several lines.

```python
result = [
    normalize(item)
    for item in items
    if is_valid(item)
]
```

## Anti-Patterns

| Avoid | Prefer |
|---|---|
| `[print(x) for x in items]` | `for x in items: print(x)` |
| `[x for x in items]` | `list(items)` |
| `sum([n * n for n in nums])` | `sum(n * n for n in nums)` |
| `list(map(lambda n: n * 2, nums))` | `[n * 2 for n in nums]` |
| Three nested `for` clauses in one line | A loop or a helper function |
| Using `:=` to be clever | A separate assignment |

## Best Practices Checklist

- Use a comprehension to **build** a value.
- Use a generator expression when the result is consumed once.
- Extract complex conditions into functions.
- Format long comprehensions on several lines.
- Do not use comprehensions for side effects.
- Measure before optimizing.

## Source

https://docs.python.org/3/library/timeit.html

https://peps.python.org/pep-0709/

https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions

In [ ]:
import sys
import timeit

nums = list(range(1000))

# Loop vs comprehension vs map: measure, do not guess
def with_loop():
    result = []
    for n in nums:
        result.append(n * 2)
    return result

def with_comprehension():
    return [n * 2 for n in nums]

def with_map():
    return list(map(lambda n: n * 2, nums))

assert with_loop() == with_comprehension() == with_map()

for func in (with_loop, with_comprehension, with_map):
    seconds = timeit.timeit(func, number=500)
    print(f"{func.__name__:<20} {seconds:.4f} s")

# Memory: a list stores everything, a generator stores only its state
big_list = [n * n for n in range(100_000)]
big_gen = (n * n for n in range(100_000))
print(sys.getsizeof(big_list) > sys.getsizeof(big_gen))
print(sum(n * n for n in range(100_000)) == sum(big_list))

# Readability: move a complex test into a named function
def is_valid(item):
    return isinstance(item, str) and item.strip() != ""

def normalize(item):
    return item.strip().lower()

items = ["  Alpha ", "", "BETA", 42, "  "]

result = [
    normalize(item)
    for item in items
    if is_valid(item)
]
print(result)

# Anti-patterns and their better forms
data = [1, 2, 3]

copy_bad = [x for x in data]
copy_good = list(data)
print(copy_bad == copy_good)

total_bad = sum([n * n for n in data])       # builds a temporary list
total_good = sum(n * n for n in data)        # no temporary list
print(total_bad == total_good)

# Side effects belong in a regular loop
for n in data:
    print("item:", n)